[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_porte_logiche_didattica_bias.ipynb)

# Un neurone (con bias) impara la porta NOR

## Cosa è successo nel notebook precedente

- Per OR e AND il neurone semplice $\hat{y} = w_1 x_1 + w_2 x_2$ funziona.
- Per **NOR** si schianta: la loss resta a 1, il gradiente è $(0,0)$ fin da subito, i pesi non si muovono.
- Motivo: in $(0,0)$ il modello vale **necessariamente** 0, ma il bersaglio NOR in $(0,0)$ è 1. Senza una manopola in più, non c'è modo di sollevare l'uscita.

## La manopola in più: il bias

Aggiungiamo un terzo parametro $b$ — il **bias** — che si somma sempre, indipendentemente dagli input:

$$\hat{y} \;=\; w_1\,x_1 + w_2\,x_2 + b$$

Geometricamente: prima la "retta-modello" era costretta a valere 0 nell'origine. Adesso può valere $b$ — qualunque numero. Da trasformazione **lineare** siamo passati a trasformazione **affine**.

Tutto il resto (loss, gradient descent, loop) **non cambia**. Cambia solo `predict` (un termine in più) e il numero di parametri (tre invece di due).

## Setup

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad
import numpy as np
import matplotlib.pyplot as plt

## 1. I dati — la porta NOR

*Posso uscire con gli amici (output = 1) solo se non ho compiti ($x_1 = 0$) e non devo aiutare in casa ($x_2 = 0$).*

| $x_1$ | $x_2$ | $y$ (NOR) |
|:-:|:-:|:-:|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 0 |

In [ ]:
X = jnp.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y = jnp.array([
    1.0,
    0.0,
    0.0,
    0.0,
])

## 2. Il modello — neurone con bias

$$\hat{y} \;=\; w_1\,x_1 + w_2\,x_2 + b$$

Tre parametri da imparare: $w_1, w_2, b$. Li raccogliamo in un unico vettore `params = [w1, w2, b]`.

Su tutti e 4 gli esempi insieme, in forma vettoriale:

$$\hat{\mathbf{y}} \;=\; X\,\mathbf{w} + b, \qquad \mathbf{w} = \begin{pmatrix}w_1\\w_2\end{pmatrix}$$

In [ ]:
def predict(params, X):
    w = params[:2]
    b = params[2]
    return X @ w + b

## 3. La loss

Identica al notebook precedente — somma dei quadrati degli errori.

$$L(\mathbf{w}, b) \;=\; \sum_{i=1}^{4} \big(y_i - \hat{y}_i\big)^2$$

In [ ]:
def loss(params, X, y):
    y_hat = predict(params, X)
    return jnp.sum((y - y_hat) ** 2)

params = jnp.array([0.0, 0.0, 0.0])    # parametri iniziali: w1, w2, b
print('L iniziale =', float(loss(params, X, y)))

## 4. Il gradiente — calcolato da JAX

Adesso il gradiente è un vettore di **tre** componenti, una per parametro:

$$\nabla L \;=\; \left(\frac{\partial L}{\partial w_1},\; \frac{\partial L}{\partial w_2},\; \frac{\partial L}{\partial b}\right)$$

Per JAX nessuna differenza: `grad(loss)` produce la nuova funzione che lo calcola, qualunque sia il numero di parametri.

In [ ]:
grad_loss = grad(loss)

g = grad_loss(params, X, y)
print('gradiente in (0, 0, 0):', g)

Le prime due componenti sono **zero**, la terza vale $-2$. Significa che, partendo da pesi nulli, $w_1$ e $w_2$ non si muovono ancora — ma $b$ sì: cresce per coprire l'errore in $(0,0)$. Questa è esattamente la "manopola" che mancava prima.

## 5. Il loop di addestramento

Identico al notebook precedente:

$$\text{params} \;\leftarrow\; \text{params} \;-\; \eta\,\nabla L$$

Stampiamo loss, parametri e gradiente ogni 50 epoche.

In [ ]:
params   = jnp.array([0.0, 0.0, 0.0])    # w1, w2, b
lr       = 0.1
n_epoche = 500

storia_loss = []

print(f"{'epoca':>6} | {'loss':>8} | {'w1':>8} {'w2':>8} {'b':>8} | {'dL/dw1':>9} {'dL/dw2':>9} {'dL/db':>9}")
print('-' * 84)

for epoca in range(n_epoche):
    perdita = loss(params, X, y)
    g       = grad_loss(params, X, y)
    storia_loss.append(float(perdita))

    if epoca % 50 == 0:
        print(f"{epoca:>6d} | {float(perdita):>8.4f} | {float(params[0]):>8.4f} {float(params[1]):>8.4f} {float(params[2]):>8.4f} | {float(g[0]):>9.4f} {float(g[1]):>9.4f} {float(g[2]):>9.4f}")

    params = params - lr * g

print('-' * 84)
print(f"{'fine':>6} | {float(loss(params, X, y)):>8.4f} | {float(params[0]):>8.4f} {float(params[1]):>8.4f} {float(params[2]):>8.4f}")

Adesso la loss **scende**. I pesi convergono a $w_1 = w_2 \approx -0.5$ e $b \approx 0.75$. Il bias positivo solleva l'uscita, i pesi negativi la abbassano quando gli input sono attivi: esattamente quello che serve a NOR.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(storia_loss, color='#38bdf8')
plt.xlabel('epoca'); plt.ylabel('loss'); plt.title('La loss diminuisce a ogni epoca')
plt.grid(alpha=0.3); plt.show()

## 6. Le predizioni del neurone addestrato

In [ ]:
y_hat = predict(params, X)

print(f"{'x1':>3} {'x2':>3} {'target':>8} {'pred':>10} {'arrotondato':>12}")
print('-' * 40)
for (x_in, t, p) in zip(X, y, y_hat):
    print(f"{int(x_in[0]):>3} {int(x_in[1]):>3} {int(t):>8} {float(p):>10.4f} {int(round(float(p))):>12}")

Le predizioni sono $\{0.75,\;0.25,\;0.25,\;-0.25\}$. **Arrotondate a 0 o 1** (soglia 0.5) restituiscono $\{1, 0, 0, 0\}$ — esattamente NOR. Il neurone, col bias, ce l'ha fatta.

## 7. Il confine di decisione

Adesso il confine è la retta $w_1 x_1 + w_2 x_2 + b = 0.5$, che **non passa più per l'origine**. Il bias l'ha staccata dallo zero — ed è esattamente quello che serviva a NOR per separare $(0,0)$ dagli altri tre punti.

In [ ]:
xx, yy_g = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
zz = float(params[0]) * xx + float(params[1]) * yy_g + float(params[2])

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(xx, yy_g, zz, levels=[-10, 0.5, 10], colors=['#fee2e2', '#dcfce7'], alpha=0.8)
ax.contour(xx, yy_g, zz, levels=[0.5], colors='black', linewidths=1.5)

for (x1, x2), label in zip(np.array(X), np.array(y)):
    color = '#10b981' if label == 1 else '#ef4444'
    ax.scatter(x1, x2, s=300, c=color, edgecolor='black', zorder=3, linewidth=2)
    ax.annotate(f'  ({int(x1)},{int(x2)}) -> {int(label)}', (x1, x2), fontsize=11, va='center')

ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title(f'Confine di decisione  ({float(params[0]):.2f} x1 + {float(params[1]):.2f} x2 + {float(params[2]):.2f} = 0.5)')
ax.grid(alpha=0.3); ax.set_aspect('equal')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
plt.show()

Il punto verde $(0,0)$ è ora isolato dagli altri tre punti rossi da una retta che taglia l'angolo in basso a sinistra. Senza bias quella retta sarebbe stata costretta a passare per $(0,0)$ — impossibile separarli.

## Riepilogo

| Concetto | Realizzazione |
|---|---|
| modello | `predict(params, X)` — $X\mathbf{w} + b$ (affine) |
| parametri | `params = [w1, w2, b]` — tre numeri |
| loss | `loss(params, X, y)` — somma dei quadrati |
| gradiente | `grad(loss)` — JAX deriva su tutti i parametri |
| algoritmo | `params = params - lr * grad_loss(params, X, y)` |

Cambiando da neurone *lineare* a neurone *affine* (un parametro in più) abbiamo sbloccato NOR. **L'algoritmo è esattamente lo stesso** del notebook precedente.

## Esercizi — modifica i `target` e rilancia

Adesso il modello è generale: con il bias, qualunque porta logica linearmente separabile è alla portata. Verifichiamolo.

### 1. Porta OR

Cambia `y` con i target di OR: $\{0, 1, 1, 1\}$. Rilancia. La loss arriva a $0.25$, le predizioni continue sono $\{0.25,\;0.75,\;0.75,\;1.25\}$ — arrotondate $\{0,1,1,1\}$ ✓. Confronta con il notebook senza bias: la loss scendeva solo a $1/3 \approx 0.33$.

### 2. Porta AND

Cambia `y` con i target di AND: $\{0, 0, 0, 1\}$. Rilancia. Stessa storia: loss $0.25$, predizioni $\{-0.25,\;0.25,\;0.25,\;0.75\}$ — arrotondate $\{0,0,0,1\}$ ✓.

### 3. Porta XOR (anticipazione)

Per scrupolo prova anche XOR: $\{0, 1, 1, 0\}$. Vedrai che la loss si ferma in alto e nessuna scelta di $w_1, w_2, b$ riesce a far meglio. Il confine di decisione è una **retta**, ma per XOR servirebbe una curva (o due rette). Il bias non basta — serve una **rete** di più neuroni. Lo vediamo nella prossima lezione.